 Binary classification NN - strategy: Make one big dataset with 0 for missing values, train weights for all N
 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset
from torch import manual_seed, nn, no_grad, optim
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

torch.manual_seed(10101010) # because we randomly sort the data inside the data loader

In [2]:
class TensorData(Dataset):
    def __init__(self, input_tensor, label_tensor):
        self.input = input_tensor
        self.labels = label_tensor

    def __len__(self):
        return self.input.size()[0]

    def __getitem__(self, index):
        return self.input[index], self.labels[index]

In [3]:
def load_data():
    input_data_5 = "kaggle_train_5_fences.csv"
    input_data_7 = "kaggle_train_7_fences.csv"
    input_data_9 = "kaggle_train_9_fences.csv"
    test_input = "kaggle_hidden_test_fences.csv"
    return {"input_data_5": pd.read_csv(input_data_5),"input_data_7": pd.read_csv(input_data_7),"input_data_9": pd.read_csv(input_data_9), "test_input": pd.read_csv(test_input)}

data = load_data()

#print(data["input_data_7"].head())

Fillnan and dummy values + only columns

In [4]:
def extract_inputs(data_input):
    # Initialize data array as a lst
    pure_data = []

    data = data_input
    data5 = data["input_data_5"].copy()
    data7 = data["input_data_7"].copy()
    data9 = data["input_data_9"].copy()
    data_test = data["test_input"].copy()

    # Insert dummy columns
    for n in ["8","7","6","5"]:
        data5.insert(6,n,0)

    for n in ["8","7"]:
        data7.insert(8,n,0)

    # Extract only columns
    data_new = {"data5" : data5.loc[:, ['0','1', '2',"3","4","5","6","7","8"]],
                "data7" : data7.loc[:, ['0','1', '2',"3","4","5","6","7","8"]],
                "data9" : data9.loc[:, ['0','1', '2',"3","4","5","6","7","8"]]}
        
    # Replace NaN with zeros in testing data set
    data_test = data_test.fillna(0)

    # Return data_train as pdf and data_test as an numpy array
    return data_new, data_test.loc[:, ['0','1', '2',"3","4","5","6","7","8"]].to_numpy()

def extract_labels(input):
    # Initialize labels list
    pure_label = []

    data = input
    data5 = data["input_data_5"].copy()
    data7 = data["input_data_7"].copy()
    data9 = data["input_data_9"].copy()
    data_test = data["test_input"].copy()
    labels_list = []

    data_new = {"data5" : data5, "data7" : data7, "data9" : data9}

    # Extract labels into a list
    for data_set in data_new.values():
        pure_label.append(data_set.loc[:,["CE"]])
    
    # Return only labels from train data, test data contains no CE, it must be determined for it
    return np.vstack(pure_label)



Symmetry funcs: i) biggest fence at bottom ii) neighboring largest fence on the right
* cyclic_sort lowkey not needed since our system is not a vector (no direction of area or center)
  

In [5]:
def cyclic_sort(data_values_large):
    # Set largest fence to bottom (= position 1)
    # Initialize for return
    rotated_arrs = []
    
    for data_set in data_values_large.values():
        arr = data_set.to_numpy().copy()
        
        for i,row in enumerate(arr):
            max_index = row.argmax()
            arr[i] = np.roll(row, shift=-max_index)
            
        rotated_arrs.append(arr)
                
    return np.vstack(rotated_arrs)
    

def orientation_sort(rotated_arrs):
    # Set to position 2, the larger of the fences neighbouring position 1
    values = rotated_arrs.copy()

    for idx, row in enumerate(values):
        right_neighbor = row[1]
        left_neighbor = row[-1]

        if left_neighbor > right_neighbor:
            values[idx, 1:] = np.flip(values[idx, 1:])

    return values

def disambiguate(data_input):
    no_cyclic = cyclic_sort(data_input)
    no_sym = orientation_sort(no_cyclic)
    return no_sym

Prepare data: i) Make dimensionless
* no need for d1h and then division by area since this could hint at: small area = not enclosed?, so the model could cheat
  
* By just removing the scale effect, only the geometry relationship is left for the NN
* therefore just divide each fence by the first (largest) fence

In [6]:
def remove_scale(data_with_scale):
    # Divide every fence length by longest fence for [-]
    # Must be done for test data also, so argmax is used

    values = data_with_scale.copy()
    for idx, row in enumerate(values):
        max_index = row.argmax()
        values[idx] = row / row[max_index]
        
    return values

Turn to tensors

In [7]:
# Load fences
data = load_data()

# Extract columns only
input_data_ordered, test_data_ordered = extract_inputs(data)
input_labels_ordered = extract_labels(data)

# Apply disambiguation
data_with_symmetry = disambiguate(input_data_ordered)

# Make dimensionless
processed_data = remove_scale(data_with_symmetry)
processed_test = remove_scale(test_data_ordered)

# Tensors for pytorch NN
train_data = torch.tensor(processed_data)
train_labels = torch.tensor(input_labels_ordered, dtype=torch.float32)

In [8]:
#print(input_data_ordered.shape) # (15000, 9)
#print(test_data_ordered.shape) # (45000, 9)
#print(input_labels_ordered.shape) # (15000, 1)
#print(input_labels_ordered) # 0 and 1
#print(input_labels_ordered[:10]) # matching excel
#print(train_data)
#print(train_labels)
print(train_data.shape)
print(train_labels.shape)

torch.Size([15000, 9])
torch.Size([15000, 1])


Setup NN - Stuff below is not done yet

In [9]:
def make_model_binary():
    return nn.Sequential(
        nn.Linear(8, 4, bias=True),
        nn.ReLU(),
        nn.Linear(8, 1, bias=True),
        nn.Sigmoid(),
    )

In [10]:
def binary_accuracy(predictions, labels):
 #
    return 

NN from lab5

In [11]:
def train_model(
    train_data,
    test_input,
    test_labels,
    model,
    loss_fn,
    accuracy_fn,
    epochs=10,
    lr=0.01,
    batch_size=1,
    print_every=1,
):
    torch.manual_seed(10101010)
    loss_dict = {"train": [], "test": [], "test_acc": []}

    # Initialize optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # We use a `DataLoader` to get batching for free!
    train_data_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, generator=torch.Generator(device=device))

    # Print header.
    print(f"Epoch    Train loss      Test loss       Test accuracy")

    for epoch in range(epochs):
        epoch_loss_sum = 0

        for x_batch, y_batch in train_data_loader:
            # Reset optimizer gradients.
            optimizer.zero_grad()

            # Predict the output
            y_pred = model(x_batch)

            # Compute the loss
            loss = loss_fn(y_pred, y_batch)
            epoch_loss_sum += loss.item()

            # Compute gradients according to newly computed loss.
            loss.backward()

            # Update the model parameters.
            optimizer.step()

        loss_dict["train"].append(epoch_loss_sum / len(train_data_loader))

        with no_grad():
            test_pred = model(test_input)
            test_loss = loss_fn(test_pred, test_labels)
            loss_dict["test"].append(test_loss.item())

            test_accuracy = accuracy_fn(test_pred, test_labels)
            loss_dict["test_acc"].append(test_accuracy.item())

        if (epoch + 1) % print_every == 0:
            print(
                f"{epoch+1: <7}  {loss_dict['train'][-1]: <14.6e}  {loss_dict['test'][-1]: <13.6e}  {loss_dict['test_acc'][-1]: .6}"
            )

    return model, loss_dict

In [12]:
# Define model arguments
total_epochs = 10
print_every = 1
batch_size = 45
lr = 1e-2
loss_fn = nn.MSELoss()

In [13]:
model = create_model()


model, loss_dict = train_model(
    train_data,
    data["x_test"],
    data["y_test"],
    model,
    loss_fn,
    epochs=total_epochs,
    lr=lr,
    batch_size=batch_size,
    print_every=print_every,
)

NameError: name 'create_model' is not defined

In [13]:
model = create_model()


model, loss_dict = train_model(
    train_data,
    data["x_test"],
    data["y_test"],
    model,
    loss_fn,
    epochs=total_epochs,
    lr=lr,
    batch_size=batch_size,
    print_every=print_every,
)

NameError: name 'create_model' is not defined